In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DoubleType
from datetime import datetime


schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])


initial_data = [
    (1001, 501, 2001, 2, 100.0, "Pending", datetime.strptime("2026-09-15", "%Y-%m-%d").date(), datetime.strptime("2026-09-15 10:00:00", "%Y-%m-%d %H:%M:%S")),
    (1002, 502, 2002, 1, 250.0, "Shipped", datetime.strptime("2026-09-15", "%Y-%m-%d").date(), datetime.strptime("2026-09-15 11:00:00", "%Y-%m-%d %H:%M:%S")),
    (1003, 503, 2003, 3, 75.0, "Delivered", datetime.strptime("2026-09-16", "%Y-%m-%d").date(), datetime.strptime("2026-09-16 09:30:00", "%Y-%m-%d %H:%M:%S")),
    (1004, 504, 2004, 1, 500.0, "Pending", datetime.strptime("2026-09-16", "%Y-%m-%d").date(), datetime.strptime("2026-09-16 12:00:00", "%Y-%m-%d %H:%M:%S")),
    (1005, 505, 2005, 4, 50.0, "Shipped", datetime.strptime("2026-09-17", "%Y-%m-%d").date(), datetime.strptime("2026-09-17 14:00:00", "%Y-%m-%d %H:%M:%S"))
]

df_initial = spark.createDataFrame(initial_data, schema=schema)


df_initial.write.format("delta").mode("overwrite").saveAsTable("orders")


display(spark.sql("SELECT * FROM orders"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at
1001,501,2001,2,100.0,Pending,2026-09-15,2026-09-15T10:00:00.000Z
1002,502,2002,1,250.0,Shipped,2026-09-15,2026-09-15T11:00:00.000Z
1003,503,2003,3,75.0,Delivered,2026-09-16,2026-09-16T09:30:00.000Z
1004,504,2004,1,500.0,Pending,2026-09-16,2026-09-16T12:00:00.000Z
1005,505,2005,4,50.0,Shipped,2026-09-17,2026-09-17T14:00:00.000Z


In [0]:
new_data = [
    (1006, 506, 2006, 2, 120.0, "Pending", datetime.strptime("2026-09-18", "%Y-%m-%d").date(), datetime.strptime("2026-09-18 10:00:00", "%Y-%m-%d %H:%M:%S")),
    (1007, 507, 2007, 1, 300.0, "Pending", datetime.strptime("2026-09-18", "%Y-%m-%d").date(), datetime.strptime("2026-09-18 11:00:00", "%Y-%m-%d %H:%M:%S"))
]

df_new = spark.createDataFrame(new_data, schema=schema)
df_new.write.format("delta").mode("append").saveAsTable("orders")

print("Total rows:", spark.sql("SELECT COUNT(*) FROM orders").collect()[0][0])

Total rows: 7


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "orders")

deltaTable.update(
    condition="order_id IN (1001, 1004)",
    set={
        "status": "CASE WHEN order_id = 1001 THEN 'Shipped' ELSE 'Cancelled' END"
    }
)

display(spark.sql("SELECT * FROM orders WHERE order_id IN (1001, 1004)"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at
1001,501,2001,2,100.0,Shipped,2026-09-15,2026-09-15T10:00:00.000Z
1004,504,2004,1,500.0,Cancelled,2026-09-16,2026-09-16T12:00:00.000Z


In [0]:
deltaTable.delete("order_id = 1004")
display(spark.sql("SELECT * FROM orders WHERE order_id = 1004"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at


In [0]:
display(spark.sql("SELECT * FROM orders ORDER BY order_id"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at
1001,501,2001,2,100.0,Shipped,2026-09-15,2026-09-15T10:00:00.000Z
1002,502,2002,1,250.0,Shipped,2026-09-15,2026-09-15T11:00:00.000Z
1003,503,2003,3,75.0,Delivered,2026-09-16,2026-09-16T09:30:00.000Z
1005,505,2005,4,50.0,Shipped,2026-09-17,2026-09-17T14:00:00.000Z
1006,506,2006,2,120.0,Pending,2026-09-18,2026-09-18T10:00:00.000Z
1007,507,2007,1,300.0,Pending,2026-09-18,2026-09-18T11:00:00.000Z


In [0]:
batch_data = [
    (1002, 502, 2002, 1, 250.0, "Delivered", datetime.strptime("2026-09-15", "%Y-%m-%d").date(), datetime.strptime("2026-09-19 09:00:00", "%Y-%m-%d %H:%M:%S")),
    (1003, 503, 2003, 5, 75.0, "Delivered", datetime.strptime("2026-09-16", "%Y-%m-%d").date(), datetime.strptime("2026-09-19 10:00:00", "%Y-%m-%d %H:%M:%S")),
    (1008, 508, 2008, 2, 90.0, "Pending", datetime.strptime("2026-09-19", "%Y-%m-%d").date(), datetime.strptime("2026-09-19 11:00:00", "%Y-%m-%d %H:%M:%S"))
]

df_batch = spark.createDataFrame(batch_data, schema=schema)

deltaTable.alias("target").merge(
    df_batch.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdate(set={
    "quantity": "source.quantity",
    "status": "source.status",
    "updated_at": "source.updated_at"
}).whenNotMatchedInsert(values={
    "order_id": "source.order_id",
    "customer_id": "source.customer_id",
    "product_id": "source.product_id",
    "quantity": "source.quantity",
    "price": "source.price",
    "status": "source.status",
    "order_date": "source.order_date",
    "updated_at": "source.updated_at"
}).execute()

display(spark.sql("SELECT * FROM orders ORDER BY order_id"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at
1001,501,2001,2,100.0,Shipped,2026-09-15,2026-09-15T10:00:00.000Z
1002,502,2002,1,250.0,Delivered,2026-09-15,2026-09-19T09:00:00.000Z
1003,503,2003,5,75.0,Delivered,2026-09-16,2026-09-19T10:00:00.000Z
1005,505,2005,4,50.0,Shipped,2026-09-17,2026-09-17T14:00:00.000Z
1006,506,2006,2,120.0,Pending,2026-09-18,2026-09-18T10:00:00.000Z
1007,507,2007,1,300.0,Pending,2026-09-18,2026-09-18T11:00:00.000Z
1008,508,2008,2,90.0,Pending,2026-09-19,2026-09-19T11:00:00.000Z


In [0]:
try:
    bad_data = [(1009, 509, 2009, "two", 100.0, "Pending", datetime.strptime("2026-09-19", "%Y-%m-%d").date(), datetime.strptime("2026-09-19 12:00:00", "%Y-%m-%d %H:%M:%S"))]
    bad_schema = StructType([
        StructField("order_id", IntegerType(), True),
        StructField("customer_id", IntegerType(), True),
        StructField("product_id", IntegerType(), True),
        StructField("quantity", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("status", StringType(), True),
        StructField("order_date", DateType(), True),
        StructField("updated_at", TimestampType(), True)
    ])
    df_bad = spark.createDataFrame(bad_data, schema=bad_schema)
    df_bad.write.format("delta").mode("append").saveAsTable("orders")
except Exception as e:
    print("Error caught successfully:", e)

Error caught successfully: [CAST_INVALID_INPUT] The value 'two' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018

JVM stacktrace:
org.apache.spark.SparkNumberFormatException
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:205)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:57)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2(Cast.scala:1520)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2$adapted(Cast.scala:1520)
	at org.apache.spark.sql.catalyst.expressions.Cast.buildCast(Cast.scala:1118)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$1(Cast.scala:152

In [0]:
discount_data = [(1009, 509, 2009, 1, 200.0, "Pending", datetime.strptime("2026-09-19", "%Y-%m-%d").date(), datetime.strptime("2026-09-19 12:00:00", "%Y-%m-%d %H:%M:%S"), 20.0)]

discount_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("discount", DoubleType(), True)
])

df_discount = spark.createDataFrame(discount_data, schema=discount_schema)

df_discount.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("orders")

display(spark.sql("SELECT * FROM orders"))

order_id,customer_id,product_id,quantity,price,status,order_date,updated_at,discount
1001,501,2001,2,100.0,Shipped,2026-09-15,2026-09-15T10:00:00.000Z,null
1005,505,2005,4,50.0,Shipped,2026-09-17,2026-09-17T14:00:00.000Z,null
1006,506,2006,2,120.0,Pending,2026-09-18,2026-09-18T10:00:00.000Z,null
1007,507,2007,1,300.0,Pending,2026-09-18,2026-09-18T11:00:00.000Z,null
1002,502,2002,1,250.0,Delivered,2026-09-15,2026-09-19T09:00:00.000Z,null
1003,503,2003,5,75.0,Delivered,2026-09-16,2026-09-19T10:00:00.000Z,null
1008,508,2008,2,90.0,Pending,2026-09-19,2026-09-19T11:00:00.000Z,null
1009,509,2009,1,200.0,Pending,2026-09-19,2026-09-19T12:00:00.000Z,20.0


In [0]:
spark.sql("DESCRIBE HISTORY orders").show(truncate=False)

+-------+-------------------+--------------+------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("orders")
df_current = spark.read.format("delta").table("orders")

print("Orders present in Version 0 but deleted in Current version:")
display(df_v0.join(df_current, "order_id", "left_anti"))

Orders present in Version 0 but deleted in Current version:


order_id,customer_id,product_id,quantity,price,status,order_date,updated_at
1004,504,2004,1,500.0,Pending,2026-09-16,2026-09-16T12:00:00.000Z
